# Jejak Bara v3.0 — notebook final (data asli NASA)
**Runtime yang sedang jalan:** tempel sel-sel ini ke notebook Anda. Label tiap sel:
- ✅ **JANGAN RUN** — sudah jalan, variabelnya masih di memori
- ▶️ **RUN** — jalankan

**Urutan RUN sekarang:** Sel 2 → Sel 10 (lama) → unggah `khg.geojson` ke Drive → Sel 7 → Sel 11 → Sel 12 → Sel 13

Data NASA: NISAR L2 GCOV (ASF DAAC) · VIIRS 375 m active fire (FIRMS) · MERRA-2 AOD `M2TMNXAER` (GES DISC) · NASA GIBS/Worldview.

## Sel 1 — Paket  ✅ JANGAN RUN

In [ ]:
!pip install -q earthaccess rasterio rioxarray geopandas xarray h5netcdf h5py scikit-image netCDF4

## Sel 2 — Drive dan modul  ▶️ RUN (aman, hanya memuat ulang jejak_bara.py)
Output harus: `versi 3.0 | gabung_frame: True`

In [ ]:
from google.colab import drive
import os, sys, json, glob
drive.mount('/content/drive')
K = '/content/drive/MyDrive/spaceapps_nisar'
for sub in ['mentah', 'olahan', 'keluaran']:
    os.makedirs(f'{K}/{sub}', exist_ok=True)
assert os.path.exists(f'{K}/jejak_bara.py'), 'Unggah jejak_bara.py ke ' + K + ' dulu'
sys.path.insert(0, K)
import importlib, jejak_bara as jb; importlib.reload(jb)
print('modul siap, versi', jb.VERSI_KONTRAK, '| gabung_frame:', hasattr(jb, 'gabung_frame'))

## Sel 3 — Login Earthdata  ✅ JANGAN RUN

In [ ]:
import earthaccess
auth = earthaccess.login(strategy='interactive', persist=True)
print('terautentikasi:', auth.authenticated)

## Sel 4 — AOI  ✅ JANGAN RUN

In [ ]:
AOI = {k: dict(v) for k, v in jb.AOI_KANDIDAT.items()}
for k, v in AOI.items(): print(f'{k:8s} {v["bbox"]}  {v["nama"]}')

## Sel 5 — Hotspot VIIRS (FIRMS)  ✅ JANGAN RUN
Kalau runtime pernah restart: cukup jalankan baris ini sebagai gantinya:
`HOTSPOT = f'{K}/mentah/hotspot_idn_2026.csv'; import pandas as pd; hs = pd.read_csv(HOTSPOT)`

In [ ]:
import io, requests, datetime as dt, pandas as pd, getpass, time
MAP_KEY = getpass.getpass('FIRMS MAP_KEY: ').strip()
st = requests.get(f'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY={MAP_KEY}', timeout=30)
print('status MAP_KEY:', st.status_code, st.text[:200])

SP, NRT = 'VIIRS_SNPP_SP', ['VIIRS_SNPP_NRT', 'VIIRS_NOAA20_NRT']   # SP kosong -> NRT
T0, T1, LANGKAH = dt.date(2026, 5, 1), dt.date(2026, 11, 14), 5     # DAY_RANGE maksimal 5

def ambil(sumber, bbox, d):
    w, s, e, n = bbox
    url = f'https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{sumber}/{w},{s},{e},{n}/{LANGKAH}/{d}'
    r = None
    for coba in range(3):
        try:
            r = requests.get(url, timeout=120); break
        except requests.RequestException:
            time.sleep(3)
    if r is None: return None, 'koneksi gagal'
    teks = r.text.strip()
    if not r.ok or not teks.startswith('latitude'):
        return None, teks[:150]
    return pd.read_csv(io.StringIO(teks)), None

potongan, galat = [], []
for k, v in AOI.items():
    d, n_sp, n_nrt = T0, 0, 0
    while d <= T1:
        df, err = ambil(SP, v['bbox'], d)
        if df is not None and len(df):
            df['sumber'] = SP; n_sp += len(df)
        else:
            if err: galat.append(f'{k} {d} {SP}: {err}')
            df = None
            for s in NRT:
                d2, err2 = ambil(s, v['bbox'], d)
                if d2 is not None:
                    if len(d2): d2['sumber'] = s; df = d2; n_nrt += len(d2)
                    break
                galat.append(f'{k} {d} {s}: {err2}')
        if df is not None:
            df['aoi'] = k; potongan.append(df)
        d += dt.timedelta(days=LANGKAH); time.sleep(0.2)
    print(f'{k}: {n_sp:,} titik dari SP, {n_nrt:,} dari NRT')

hs = pd.concat(potongan, ignore_index=True).drop_duplicates(['latitude', 'longitude', 'acq_date', 'acq_time'])
HOTSPOT = f'{K}/mentah/hotspot_idn_2026.csv'
hs.to_csv(HOTSPOT, index=False)
print(len(hs), 'titik panas ->', HOTSPOT)
print(pd.crosstab(hs.aoi, pd.to_datetime(hs.acq_date).dt.to_period('M')))

## Sel 6 — Peta sebaran hotspot  ✅ JANGAN RUN (opsional)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, len(AOI), figsize=(5 * len(AOI), 4.5))
for a, (k, v) in zip(ax, AOI.items()):
    w, s, e, n = v['bbox']
    sub = hs[(hs.longitude.between(w, e)) & (hs.latitude.between(s, n))]
    a.scatter(sub.longitude, sub.latitude, s=1, alpha=.3, c=pd.to_datetime(sub.acq_date).map(pd.Timestamp.toordinal), cmap='plasma')
    a.set_xlim(w, e); a.set_ylim(s, n); a.set_title(f'{k}: {len(sub):,} titik panas'); a.set_aspect('equal')
plt.tight_layout(); plt.show()

## Sel 7 — KHG dan kabupaten  ▶️ RUN setelah `khg.geojson` diunggah ke `mentah/<aoi>/`
Output harus: `kalteng: 13 poligon KHG, 6 kabupaten -> OK`

In [ ]:
import geopandas as gpd
KOLOM_KAB = 'shapeName'
SUMBER_KAB = 'https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/main/releaseData/gbOpen/IDN/ADM2/geoBoundaries-IDN-ADM2_simplified.geojson'

for k, v in AOI.items():
    os.makedirs(f'{K}/mentah/{k}', exist_ok=True)
    pk = f'{K}/mentah/{k}/kabupaten.geojson'
    if not os.path.exists(pk):
        kab = gpd.read_file(SUMBER_KAB).clip(v['bbox']); kab = kab[~kab.is_empty]; kab.to_file(pk, driver='GeoJSON')
    kab = gpd.read_file(pk)
    p = f'{K}/mentah/{k}/khg.geojson'
    if os.path.exists(p):
        khg = gpd.read_file(p)
        print(f'{k}: {len(khg)} poligon KHG, {len(kab)} kabupaten  -> OK')
    else:
        print(f'{k}: KHG BELUM ADA -> unggah ke {p}')

## Sel 7b — AOD MERRA-2  ✅ JANGAN RUN (sudah selesai)

In [ ]:
TGL_ASAP = ('2026-07-01', '2026-10-31')
m2 = earthaccess.search_data(short_name='M2TMNXAER', temporal=TGL_ASAP)
f_m2 = earthaccess.download(m2, local_path='/content/merra2')
for k, v in AOI.items():
    jb.buat_aod_rerata(f_m2, v['bbox'], f'{K}/mentah/{k}/aod_merra2.tif', variabel='TOTEXTTAU')

## Sel 8 — Cari granule NISAR (semua koleksi + footprint)  ✅ JANGAN RUN

In [ ]:
import re
from shapely.geometry import Polygon, box as kotak
from shapely.ops import unary_union

def jejak_granule(umm):
    try:
        geo = umm['SpatialExtent']['HorizontalSpatialDomain']['Geometry']
        pol = [Polygon([(p['Longitude'], p['Latitude']) for p in gp['Boundary']['Points']]).buffer(0)
               for gp in geo.get('GPolygons', [])]
        pol += [kotak(b['WestBoundingCoordinate'], b['SouthBoundingCoordinate'],
                      b['EastBoundingCoordinate'], b['NorthBoundingCoordinate']) for b in geo.get('BoundingRectangles', [])]
        return unary_union(pol) if pol else None
    except Exception:
        return None

KOLEKSI = ['NISAR_L2_GCOV_V1', 'NISAR_L2_GCOV_PROVISIONAL_V1', 'NISAR_L2_GCOV_BETA_V1']
T_CARI = ('2026-05-01', '2026-11-15')

def daftar_granule(bbox, t0=T_CARI[0], t1=T_CARI[1]):
    rows = []
    for prio, sn in enumerate(KOLEKSI):
        try:
            hasil = earthaccess.search_data(short_name=sn, bounding_box=bbox, temporal=(t0, t1), count=2000)
        except Exception as ex:
            print(f'   {sn}: gagal ({ex})'); continue
        print(f'   {sn}: {len(hasil)} granule')
        for g in hasil:
            nama = g['umm']['GranuleUR']; b = nama.split('_')
            url = next((u for u in g.data_links(access='external') if u.endswith('.h5')), None)
            tgl = re.search(r'_(\d{8})T\d{6}_', nama)
            if not (tgl and url and len(b) > 9): continue
            rows.append(dict(tanggal=f'{tgl.group(1)[:4]}-{tgl.group(1)[4:6]}-{tgl.group(1)[6:]}',
                             track=b[5], arah=b[6], frame=b[7], pol=b[9], koleksi=sn, prio=prio, nama=nama, url=url,
                             jejak=jejak_granule(g['umm'])))
    df = pd.DataFrame(rows, columns=['tanggal','track','arah','frame','pol','koleksi','prio','nama','url','jejak'])
    return (df.sort_values('prio').drop_duplicates(['tanggal','track','arah','frame','pol'])
              .sort_values(['tanggal','track','frame']).reset_index(drop=True))

GRAN = {}
for k, v in AOI.items():
    print(f'\n=== {k} ===')
    GRAN[k] = daftar_granule(v['bbox'])

## Sel 9 — Pilih lintasan + pra/pasca  ✅ JANGAN RUN
(Kalau dijalankan ulang, WAJIB jalankan sel 9b lagi sesudahnya.)

In [ ]:
import numpy as np
PILIH = {}
for k, df in GRAN.items():
    dh = df[df.pol.str.startswith(('DH', 'QP'))]
    a = kotak(*AOI[k]['bbox'])
    def cak(g):
        return g.groupby('tanggal')['jejak'].apply(lambda s: unary_union([x for x in s if x is not None]).intersection(a).area / a.area
                                                   if any(x is not None for x in s) else np.nan).median()
    kand = sorted([(cak(g), g.tanggal.nunique(), tr, ar) for (tr, ar), g in dh.groupby(['track', 'arah']) if g.tanggal.nunique() >= 2],
                  key=lambda x: (np.nan_to_num(x[0], nan=-1), x[1]), reverse=True)
    c, nt, tr, ar = kand[0]
    g = dh[(dh.track == tr) & (dh.arah == ar)].copy(); tgl = sorted(g.tanggal.unique())
    w, s_, e, n = AOI[k]['bbox']
    h = hs[hs.longitude.between(w, e) & hs.latitude.between(s_, n)]
    h = h[h.confidence.astype(str).str.lower().isin(['n', 'h'])]
    d = pd.to_datetime(h.acq_date).sort_values()
    mulai, reda = d.quantile(0.10).strftime('%Y-%m-%d'), d.quantile(0.90).strftime('%Y-%m-%d')
    pra = max([t for t in tgl if t <= mulai], default=tgl[0]); pasca = min([t for t in tgl if t >= reda], default=tgl[-1])
    PILIH[k] = dict(granule=g, track=tr, arah=ar, pra=pra, pasca=pasca, folder=f'{K}/mentah/{k}/gcov/T{tr}{ar}')
    print(f'{k}: track {tr} {ar} | cakupan ~{c:.0%} | pra {pra} -> pasca {pasca}')

## Sel 9b — Override tanggal  ✅ JANGAN RUN (sudah dijalankan)

In [ ]:
PILIH['kalteng']['pra'] = '2026-07-24'     # sebelum lonjakan hotspot Agustus
PILIH['merauke']['pasca'] = '2026-11-01'   # Oktober masih banyak hotspot
for k in ['kalteng', 'merauke', 'riau']:
    p = PILIH[k]; print(k, p['track'], p['arah'], p['pra'], '->', p['pasca'])

## Sel 10 — Unduh + potong + gabung frame NISAR  ▶️ RUN SEKARANG (paling lama, tidak butuh KHG)
Kalau ada `GAGAL`, jalankan ulang sel ini saja — yang sudah selesai dilewati.
Untuk AOI berikutnya: ganti `PROSES` lalu run lagi.

In [ ]:
import shutil, time, concurrent.futures as cf, rasterio, numpy as np
PROSES = ['kalteng']      # lalu ['merauke'], lalu ['riau']
RESOLUSI_SUBSET = 80
PARALEL = 3
LOKAL = '/content/tmp_gcov'; os.makedirs(LOKAL, exist_ok=True)
print('disk lokal bebas: %.0f GB' % (shutil.disk_usage('/content').free / 1e9))

def potong_satu(k, r, tmp):
    f = earthaccess.download([r.url], local_path=f'{tmp}/h5_{r.frame}', threads=4)[0]
    try:
        return jb.subset_gcov(str(f), AOI[k]['bbox'], tmp, f'f{r.frame}', resolusi_m=RESOLUSI_SUBSET)
    finally:
        shutil.rmtree(f'{tmp}/h5_{r.frame}', ignore_errors=True)

def kerjakan_tanggal(k, tgl, rows):
    out = PILIH[k]['folder']; os.makedirs(out, exist_ok=True)
    if all(os.path.exists(f'{out}/{tgl}_{p}.tif') for p in ('hv', 'hh')):
        return f'{k} {tgl}: sudah ada, dilewati'
    t0 = time.time(); tmp = f'{LOKAL}/{k}_{tgl}'; os.makedirs(tmp, exist_ok=True)
    try:
        hasil = []
        for _, r in rows.iterrows():
            try: hasil.append(potong_satu(k, r, tmp))
            except ValueError: print(f'   {tgl} frame {r.frame}: tidak beririsan AOI, dilewati')
        if not hasil: return f'{k} {tgl}: tidak ada frame yang beririsan AOI'
        for L, p in (('HVHV', 'hv'), ('HHHH', 'hh')):
            jb.gabung_frame([h.get(L) for h in hasil], f'{tmp}/{tgl}_{p}.tif')
            shutil.copyfile(f'{tmp}/{tgl}_{p}.tif', f'{out}/{tgl}_{p}.tif')
        return f'{k} {tgl}: {len(hasil)} frame digabung, {time.time()-t0:.0f} s'
    except Exception as ex:
        return f'{k} {tgl}: GAGAL {ex}'
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

for k in PROSES:
    p = PILIH[k]; g = p['granule']; g = g[(g.tanggal >= p['pra']) & (g.tanggal <= p['pasca'])]
    per_tgl = list(g.groupby('tanggal'))
    print(f'== {k}: track {p["track"]} {p["arah"]}, {len(per_tgl)} tanggal')
    with cf.ThreadPoolExecutor(PARALEL) as ex:
        for msg in ex.map(lambda t: kerjakan_tanggal(k, t[0], t[1]), per_tgl):
            print('  ', msg)
    w, s, e, n = AOI[k]['bbox']; luas = (e - w) * (n - s) * 111 * 111
    for f in sorted(glob.glob(f'{p["folder"]}/*_hv.tif')):
        with rasterio.open(f) as r:
            a = r.read(1); v = np.isfinite(a) & (a > 0)
            print(f'   {os.path.basename(f)}: cakupan AOI ~{v.sum()*r.res[0]*r.res[1]/1e6/luas:.0%}')

## Sel 11 — Pipeline v3 (SLIC + Multi-Otsu + tracking)  ▶️ RUN setelah Sel 10 selesai DAN Sel 7 menulis OK

In [ ]:
AMBANG = {'riau': None, 'kalteng': None, 'merauke': None}   # None = Multi-Otsu otomatis
RESOLUSI = 80
JALANKAN = ['kalteng']       # lalu ['merauke'], lalu ['riau']

for k in JALANKAN:
    assert os.path.exists(f'{K}/mentah/{k}/khg.geojson'), f'KHG {k} belum ada — unggah dulu lalu run sel 7'
    p = PILIH[k]; g = p['folder']
    deret = [(t, f'{g}/{t}_hv.tif', f'{g}/{t}_hh.tif') for t in sorted(p['granule'].tanggal.unique())
             if p['pra'] <= t <= p['pasca'] and os.path.exists(f'{g}/{t}_hv.tif')]
    print(k, ':', len(deret), 'tanggal di deret')
    aod = f'{K}/mentah/{k}/aod_merra2.tif'
    jb.proses(
        aoi_nama=AOI[k]['nama'], aoi_bbox=AOI[k]['bbox'],
        tanggal_pra=p['pra'], tanggal_pasca=p['pasca'],
        hv_pra=f'{g}/{p["pra"]}_hv.tif', hv_pasca=f'{g}/{p["pasca"]}_hv.tif',
        hh_pra=f'{g}/{p["pra"]}_hh.tif', hh_pasca=f'{g}/{p["pasca"]}_hh.tif',
        khg=f'{K}/mentah/{k}/khg.geojson', hotspot_csv=HOTSPOT,
        kabupaten=f'{K}/mentah/{k}/kabupaten.geojson', kolom_kabupaten=KOLOM_KAB,
        deret=deret, aod_tif=aod if os.path.exists(aod) else None, sumber_aod='MERRA-2 M2TMNXAER TOTEXTTAU',
        granule_pra=', '.join(p['granule'][p['granule'].tanggal == p['pra']].nama),
        granule_pasca=', '.join(p['granule'][p['granule'].tanggal == p['pasca']].nama),
        produk=p['granule'].koleksi.iloc[0], mode='nisar',
        folder_keluaran=f'{K}/keluaran/{k}', folder_olahan=f'{K}/olahan/{k}',
        param=jb.Parameter(ambang_hv_db=AMBANG[k], resolusi_m=RESOLUSI, ukuran_filter=3),
    )

## Sel 12 — Cek hasil  ▶️ RUN, lalu kirim output-nya ke Claude

In [ ]:
from IPython.display import Image, display
for k in JALANKAN:
    print(k); display(Image(f'{K}/olahan/{k}/histogram_multiotsu.png')); display(Image(f'{K}/olahan/{k}/plot_asap_hotspot.png'))
    st = json.load(open(f'{K}/keluaran/{k}/statistik.json'))
    print({x: st.get(x) for x in ['total_ha','terkonfirmasi_ha','kandidat_tersembunyi_ha','perlu_verifikasi_ha',
                                 'rasio_kebutaan_termal_akibat_asap_persen','arah_rambatan_dampak_dominan',
                                 'ambang_dipakai_db','jumlah_poligon','jumlah_hotspot','rerata_aod']})

## Sel 13 — Bungkus untuk repo  ▶️ RUN setelah Sel 12 dicek
Ekstrak `keluaran.zip` menimpa folder `keluaran/` di repo → commit → push. `olahan.zip` untuk slide.

In [ ]:
import shutil
jb.tulis_indeks(f'{K}/keluaran')
shutil.make_archive('/content/keluaran', 'zip', f'{K}/keluaran')
shutil.make_archive('/content/olahan', 'zip', f'{K}/olahan')
from google.colab import files; files.download('/content/keluaran.zip'); files.download('/content/olahan.zip')
print('Ekstrak keluaran.zip menimpa folder keluaran/ di repo, lalu commit + push.')